In [1]:
# 创建代理
import os
tavily_api = os.environ.get("TAVILY_API_KEY")
tavily_api
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langchain_tavily.tavily_search import TavilySearch  # 搜索工具

tool = TavilySearch(max_results=2)
print(type(tool))
print(tool.name)


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
# 配置state


<class 'langchain_tavily.tavily_search.TavilySearch'>
tavily_search


In [2]:
# 添加持久化支持
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

In [3]:
class Agent:
    # 调用模型，检查操作存在，执行操作

    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        # 局部变量，仅存在于init内部
        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_openai)  # 创建llm的节点
        graph.add_node("action", self.take_action)

        # 条件边
        graph.add_conditional_edges(
            "llm",  # 传入的第一个参数
            self.exists_action,  # 决定走向
            {True: "action", False: END}  # 表示函数之间的映射方法
        )
        graph.add_edge("action", "llm")  # 添加边连接action和llm两个节点，action指向llm
        graph.set_entry_point("llm")  # 设置图的入口点
        self.graph = graph.compile(checkpointer=checkpointer)

        # 传入工具列表
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    # 创建条件边
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0  # 判断当前是否还需要函数调用

    # 调用模型
    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []  # 存放调用工具返回的结果
        for t in tool_calls:
            # 通常情况下模型支持并行调用
            print(f"Calling Tools:{t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=result))
        print("Back to the model")
        return {"messages": results}




In [4]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence).\
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to make \
some of the calls before asking."""

model = ChatDeepSeek(model="deepseek-v4-flash")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [5]:
messages = [HumanMessage(content="What is the weather in Sichuan")]

In [6]:
# 配置线程
thread = {"configurable": {"thread_id": 1}}

In [7]:
count = 0
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v["messages"])


[AIMessage(content="I'll look up the current weather conditions in Sichuan for you.", additional_kwargs={'refusal': None, 'reasoning_content': 'The user asks about weather in Sichuan. I should search for current weather.'}, response_metadata={'token_usage': {'completion_tokens': 178, 'prompt_tokens': 1901, 'total_tokens': 2079, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 15, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 1901}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '73663562-dc6c-4acf-8f4b-2af2a06a07df', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1e226-e0ef-7551-8b1f-eac4ada550d9-0', tool_calls=[{'name': 'tavi

In [8]:
messages = [HumanMessage(content="What about in Chongqing")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='I\'ll look that up — quick note first: Chongqing is a separate municipality (a "direct-administered city"), not part of Sichuan Province, though it was historically part of Sichuan until 1997. Let me pull the forecast.', additional_kwargs={'refusal': None, 'reasoning_content': 'The user asks about Chongqing. Note: Chongqing is a separate municipality, not part of Sichuan province. I should clarify that and search for weather.'}, response_metadata={'token_usage': {'completion_tokens': 230, 'prompt_tokens': 5864, 'total_tokens': 6094, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 31, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 1792, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 1792, 'prompt_cache_miss_tokens': 4072}, 'model_provider': 'deepseek', 'model_name': 'de

In [9]:
messages = [HumanMessage(content="Which is warmer？")]
thread = {"configurable": {"thread_id": "2"}} # 更改id然后重新运行
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Your question is missing the two things being compared — "which is warmer" of what?\n\nCould you clarify? For example, are you asking about:\n\n- **Materials:** wool vs. down, cotton vs. polyester, etc.\n- **Places/climates:** two cities, two planets, the poles vs. the equator\n- **Objects/appliances:** two jacket models, two sleeping bags, heat pumps vs. furnaces\n- **Colors/surfaces:** black vs. white in sunlight\n\nAlso helpful: **warmer in what sense** — air temperature, how well it retains body heat, surface temperature in sunlight, etc.\n\nTell me the two items and I\'ll look up the numbers and give you a comparison.', additional_kwargs={'refusal': None, 'reasoning_content': 'The user asks "Which is warmer?" — ambiguous. I need to ask a follow-up. Per instructions, I may search first if needed, but here there\'s no clear referent. I should ask a clarifying question. But the instructions say "If you need to look up some information before asking a 

In [10]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

In [11]:
conn = await aiosqlite.connect(":memory:")
memory = AsyncSqliteSaver(conn)
await memory.setup()

abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [12]:
messages = [HumanMessage(content="What is the weather in Sichuan")]
thread = {"configurable": {"thread_id": "4"}} # 更改id然后重新运行
async for event in abot.graph.astream_events({"messages": messages}, thread):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            print(content, end="|")

I|'ll| look| up| the| current| weather| in| Sichuan| for| you|.|Calling Tools:{'name': 'tavily_search', 'args': {'query': 'Sichuan weather today forecast', 'topic': 'general', 'time_range': 'day', 'search_depth': 'advanced'}, 'id': 'call_00_mO6sOUhKV7DK1BxQZqXH9905', 'type': 'tool_call'}
Calling Tools:{'name': 'tavily_search', 'args': {'query': 'Chengdu Sichuan current temperature weather conditions', 'search_depth': 'advanced'}, 'id': 'call_01_0JX9qHMekQfKJxKM4qXh4336', 'type': 'tool_call'}
Back to the model
Here|'s| the| current| weather| for| **|S|ich|uan|,| China|**| (|data| timestamp|ed| **|Nov| |16|,| |202|6|,| ~|10|:|15| AM| local| time|**|):

|##| Cheng|du| (|prov|incial| capital|)
|-| **|Current|:**| ~|12|°|C| (|53|.|6|°|F|),| feels| like| ~|11|°|C| (|51|.|4|°|F|)
|-| **|Condition|:**| Patch|y| rain| nearby| 🌦|️|
|-| **|Hum|idity|:**| |81|%| || **|Wind|:**| NE| ~|8| km|/h| (|5| mph|)
|-| **|Ch|ance| of| rain|:**| |35|%| (|76|%| for| the| day|)
|-| **|Today|'s| range|:**| |11|–